
# 12 - OLS temporal effects

This notebook studies the temporal fixed-effect axis:

> Does residual temporal heterogeneity remain after the core OLS controls?

Experiments compared:

1. `ols_core`
2. `ols_core_year_fe`
3. `ols_core_quarter_fe`
4. `ols_core_year_plus_quarter_fe`

The goal is to diagnose whether year and quarter effects improve fit, whether the gains are substantive, and whether residual errors show temporal drift or seasonality.


## 00. Setup and run discovery

In [ ]:

from pathlib import Path
import json
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/home/matias/repos/income-modeling-eph"),
]

ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "reports" / "runs").exists()),
    Path("/home/matias/repos/income-modeling-eph"),
)

RUNS_DIR = ROOT / "reports" / "runs"
DATASET_PATH = ROOT / "data" / "processed" / "modeling_dataset.parquet"

OUTPUT_DIR = ROOT / "reports" / "notebook_outputs" / "ols_temporal_effects"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

print("ROOT:", ROOT)
print("RUNS_DIR:", RUNS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


def read_json_if_exists(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def read_yaml_if_exists(path: Path):
    if not path.exists():
        return None
    return yaml.safe_load(path.read_text())


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def read_parquet_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()


RUN_PATTERNS = {
    "ols_core": "ols_core_*",
    "ols_core_year_fe": "ols_core_year_fe_*",
    "ols_core_quarter_fe": "ols_core_quarter_fe_*",
    "ols_core_year_plus_quarter_fe": "ols_core_year_plus_quarter_fe_*",
}

ORDER = [
    "ols_core",
    "ols_core_year_fe",
    "ols_core_quarter_fe",
    "ols_core_year_plus_quarter_fe",
]

DISPLAY_LABELS = {
    "ols_core": "Core OLS",
    "ols_core_year_fe": "+ Year FE",
    "ols_core_quarter_fe": "+ Quarter FE",
    "ols_core_year_plus_quarter_fe": "+ Year + Quarter FE",
}


def discover_runs() -> dict[str, Path | None]:
    runs = {}
    for experiment, pattern in RUN_PATTERNS.items():
        candidates = sorted(RUNS_DIR.glob(pattern), key=lambda p: p.name)
        if not candidates:
            runs[experiment] = None
            continue

        exact = []
        for candidate in candidates:
            config = read_yaml_if_exists(candidate / "config_used.yaml") or {}
            experiment_id = ((config.get("experiment") or {}).get("id"))
            if experiment_id == experiment:
                exact.append(candidate)

        runs[experiment] = exact[-1] if exact else candidates[-1]
    return runs


RUNS = discover_runs()

run_table = pd.DataFrame(
    [
        {"experiment": exp, "run_dir": str(path) if path else None, "found": path is not None}
        for exp, path in RUNS.items()
    ]
)

run_table


## 01. Load backend artifacts

In [ ]:

def load_model_comparison(run_dir: Path, experiment: str) -> pd.DataFrame:
    df = read_csv_if_exists(run_dir / "metrics" / "model_comparison.csv")
    if df.empty:
        return df
    df["experiment"] = experiment
    df["run_dir"] = str(run_dir)
    return df


def load_predictions(run_dir: Path, experiment: str, split: str) -> pd.DataFrame:
    path = run_dir / "predictions" / f"{split}_predictions.parquet"
    df = read_parquet_if_exists(path)
    if df.empty:
        return df

    df["experiment"] = experiment
    df["run_dir"] = str(run_dir)
    df["split"] = split

    if "residual" not in df.columns:
        df["residual"] = df["y_true"] - df["y_pred"]
    if "abs_error" not in df.columns:
        df["abs_error"] = df["residual"].abs()
    if "squared_error" not in df.columns:
        df["squared_error"] = df["residual"] ** 2

    return df


def load_run_metadata(run_dir: Path, experiment: str) -> dict:
    feature_columns = read_json_if_exists(run_dir / "feature_columns.json") or []
    dataset_card = read_json_if_exists(run_dir / "dataset_card.json") or {}
    config_used = read_yaml_if_exists(run_dir / "config_used.yaml") or {}
    feature_view = config_used.get("feature_view") or {}
    fixed_effects = (config_used.get("model_design") or {}).get("fixed_effects") or []

    return {
        "experiment": experiment,
        "run_dir": str(run_dir),
        "n_feature_columns": len(feature_columns),
        "feature_columns": feature_columns,
        "feature_view_name": feature_view.get("name"),
        "include_blocks": feature_view.get("include_blocks"),
        "fixed_effects_config": fixed_effects,
        "dataset_card": dataset_card,
        "config_used": config_used,
    }


model_comparison_parts = []
prediction_parts = []
metadata_rows = []

for experiment, run_dir in RUNS.items():
    if run_dir is None:
        continue
    model_comparison_parts.append(load_model_comparison(run_dir, experiment))
    for split in ["validation", "test"]:
        prediction_parts.append(load_predictions(run_dir, experiment, split))
    metadata_rows.append(load_run_metadata(run_dir, experiment))

model_comparison = (
    pd.concat([d for d in model_comparison_parts if not d.empty], ignore_index=True)
    if any(not d.empty for d in model_comparison_parts)
    else pd.DataFrame()
)

predictions = (
    pd.concat([d for d in prediction_parts if not d.empty], ignore_index=True)
    if any(not d.empty for d in prediction_parts)
    else pd.DataFrame()
)

run_metadata = pd.DataFrame(metadata_rows)

print("model_comparison:", model_comparison.shape)
print("predictions:", predictions.shape)
print("run_metadata:", run_metadata.shape)

run_metadata[["experiment", "feature_view_name", "include_blocks", "fixed_effects_config", "n_feature_columns"]]



## 02. Load time columns from modeling dataset

Predictions are enriched with `ANO4` and `TRIMESTRE` from the processed modeling dataset using `row_id`.


In [ ]:

if predictions.empty:
    raise RuntimeError(
        "No predictions were loaded. Run the OLS temporal experiments first or check RUN_PATTERNS."
    )

time_cols = ["row_id", "ANO4", "TRIMESTRE"]
if DATASET_PATH.exists():
    dataset_time = pd.read_parquet(DATASET_PATH, columns=time_cols)
else:
    raise FileNotFoundError(f"Processed dataset not found: {DATASET_PATH}")

pred_time = predictions.merge(dataset_time, on="row_id", how="left", validate="many_to_one")

# Normalize labels for plotting.
pred_time["ANO4"] = pred_time["ANO4"].astype("string")
pred_time["TRIMESTRE"] = pred_time["TRIMESTRE"].astype("string")

pred_time[["experiment", "split", "row_id", "ANO4", "TRIMESTRE", "y_true", "y_pred", "residual"]].head()


## 03. Aggregate metrics and comparison vs core

In [ ]:

def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    if len(y_true) == 0:
        return np.nan
    denom = np.sum((y_true - y_true.mean()) ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((y_true - y_pred) ** 2) / denom


def aggregate_metrics(g):
    y = g["y_true"].astype(float)
    yhat = g["y_pred"].astype(float)
    residual = y - yhat
    return pd.Series({
        "n": len(g),
        "r2": safe_r2(y, yhat),
        "mae": residual.abs().mean(),
        "rmse": np.sqrt((residual ** 2).mean()),
        "mean_error": (yhat - y).mean(),
        "sd_y_true": y.std(),
        "sd_y_pred": yhat.std(),
        "compression_ratio": yhat.std() / y.std() if y.std() else np.nan,
    })


metrics_by_split = (
    pred_time
    .groupby(["experiment", "split"], as_index=False)
    .apply(aggregate_metrics, include_groups=False)
    .reset_index(drop=True)
)

metrics_by_split["experiment_order"] = metrics_by_split["experiment"].map({exp: i for i, exp in enumerate(ORDER)})
metrics_by_split["label"] = metrics_by_split["experiment"].map(DISPLAY_LABELS)

validation_summary = (
    metrics_by_split
    .query("split == 'validation'")
    .sort_values("experiment_order")
    .reset_index(drop=True)
)

core = validation_summary.query("experiment == 'ols_core'")
if not core.empty:
    core = core.iloc[0]
    for col in ["r2", "mae", "rmse", "compression_ratio"]:
        validation_summary[f"delta_{col}_vs_core"] = validation_summary[col] - core[col]

metrics_by_split.to_csv(TABLE_DIR / "T1_ols_temporal_metrics_by_split.csv", index=False)
validation_summary.to_csv(TABLE_DIR / "T2_ols_temporal_validation_vs_core.csv", index=False)
run_metadata.to_csv(TABLE_DIR / "T3_ols_temporal_run_metadata.csv", index=False)

validation_summary[[
    "experiment", "label", "n", "r2", "delta_r2_vs_core",
    "mae", "delta_mae_vs_core", "rmse", "delta_rmse_vs_core",
    "compression_ratio", "delta_compression_ratio_vs_core"
]].round(5)


## 04. Figure 1 — validation ΔR² vs core

In [ ]:

plot_df = validation_summary.copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(plot_df["label"], plot_df["delta_r2_vs_core"])
ax.axhline(0, linewidth=1)
ax.set_title("Temporal FE contribution: validation ΔR² vs core OLS")
ax.set_xlabel("Specification")
ax.set_ylabel("ΔR² vs ols_core")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "F1_ols_temporal_delta_r2_vs_core.png", dpi=160)
plt.show()


## 05. Residual means by year and quarter

In [ ]:

def temporal_group_summary(df, group_col):
    return (
        df
        .groupby(["experiment", "split", group_col], as_index=False)
        .agg(
            n=("row_id", "size"),
            mean_y_true=("y_true", "mean"),
            mean_y_pred=("y_pred", "mean"),
            mean_residual=("residual", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda s: np.sqrt(np.mean(s))),
        )
        .sort_values(["experiment", "split", group_col])
    )

year_summary = temporal_group_summary(pred_time, "ANO4")
quarter_summary = temporal_group_summary(pred_time, "TRIMESTRE")
year_quarter_summary = temporal_group_summary(
    pred_time.assign(ANO4_TRIMESTRE=pred_time["ANO4"].astype(str) + "Q" + pred_time["TRIMESTRE"].astype(str)),
    "ANO4_TRIMESTRE",
)

year_summary.to_csv(TABLE_DIR / "T4_ols_temporal_residual_by_year.csv", index=False)
quarter_summary.to_csv(TABLE_DIR / "T5_ols_temporal_residual_by_quarter.csv", index=False)
year_quarter_summary.to_csv(TABLE_DIR / "T6_ols_temporal_residual_by_year_quarter.csv", index=False)

display(year_summary.query("split == 'validation'").round(4))
display(quarter_summary.query("split == 'validation'").round(4))



## 06. Figure 2 — residual mean by year

If residual means are large or patterned for `ols_core`, year FE may be absorbing systematic temporal drift.


In [ ]:

plot_df = year_summary.query("split == 'validation'").copy()

fig, ax = plt.subplots(figsize=(8, 5))
for experiment in ORDER:
    tmp = plot_df.query("experiment == @experiment").sort_values("ANO4")
    if tmp.empty:
        continue
    ax.plot(tmp["ANO4"], tmp["mean_residual"], marker="o", label=DISPLAY_LABELS.get(experiment, experiment))

ax.axhline(0, linewidth=1)
ax.set_title("Validation mean residual by year")
ax.set_xlabel("Year")
ax.set_ylabel("Mean residual: y_true - y_pred")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "F2_ols_temporal_mean_residual_by_year.png", dpi=160)
plt.show()


## 07. Figure 3 — residual mean by quarter

In [ ]:

plot_df = quarter_summary.query("split == 'validation'").copy()

fig, ax = plt.subplots(figsize=(8, 5))
for experiment in ORDER:
    tmp = plot_df.query("experiment == @experiment").sort_values("TRIMESTRE")
    if tmp.empty:
        continue
    ax.plot(tmp["TRIMESTRE"], tmp["mean_residual"], marker="o", label=DISPLAY_LABELS.get(experiment, experiment))

ax.axhline(0, linewidth=1)
ax.set_title("Validation mean residual by quarter")
ax.set_xlabel("Quarter")
ax.set_ylabel("Mean residual: y_true - y_pred")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "F3_ols_temporal_mean_residual_by_quarter.png", dpi=160)
plt.show()



## 08. Figure 4 — year-quarter residual path

This is not a second-order FE model; it is just a diagnostic path over calendar cells.


In [ ]:

plot_df = year_quarter_summary.query("split == 'validation'").copy()

fig, ax = plt.subplots(figsize=(10, 5))
for experiment in ORDER:
    tmp = plot_df.query("experiment == @experiment").sort_values("ANO4_TRIMESTRE")
    if tmp.empty:
        continue
    ax.plot(tmp["ANO4_TRIMESTRE"], tmp["mean_residual"], marker="o", label=DISPLAY_LABELS.get(experiment, experiment))

ax.axhline(0, linewidth=1)
ax.set_title("Validation mean residual by year-quarter cell")
ax.set_xlabel("Year-quarter")
ax.set_ylabel("Mean residual: y_true - y_pred")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "F4_ols_temporal_mean_residual_by_year_quarter.png", dpi=160)
plt.show()



## 09. Fixed-effect coefficient tables

If the backend wrote fixed-effect coefficients, this section loads and displays them. Otherwise, the residual-by-time tables above remain the main empirical diagnostic.


In [ ]:

def find_fe_coefficient_files(run_dir: Path) -> list[Path]:
    diagnostics_dir = run_dir / "diagnostics"
    if not diagnostics_dir.exists():
        return []
    patterns = [
        "*fixed_effect*coefficients*.csv",
        "*FixedEffect*coefficients*.csv",
        "*fe*coefficients*.csv",
    ]
    files = []
    for pattern in patterns:
        files.extend(diagnostics_dir.glob(pattern))
    return sorted(set(files))


fe_parts = []
for experiment, run_dir in RUNS.items():
    if run_dir is None:
        continue
    for path in find_fe_coefficient_files(run_dir):
        df = pd.read_csv(path)
        df["experiment"] = experiment
        df["source_file"] = str(path.relative_to(run_dir))
        fe_parts.append(df)

fe_coefficients = pd.concat(fe_parts, ignore_index=True) if fe_parts else pd.DataFrame()

if fe_coefficients.empty:
    print("No fixed-effect coefficient files found. Use residual-by-time diagnostics above.")
else:
    fe_coefficients.to_csv(TABLE_DIR / "T7_ols_temporal_fixed_effect_coefficients_raw.csv", index=False)
    display(fe_coefficients.head(50))


## 10. Automatic diagnostic summary

In [ ]:

def fmt(x, digits=5):
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"

summary = validation_summary.set_index("experiment")
notes = []

for exp in ["ols_core_year_fe", "ols_core_quarter_fe", "ols_core_year_plus_quarter_fe"]:
    if exp in summary.index and "ols_core" in summary.index:
        notes.append(
            f"{exp}: ΔR² vs core = {fmt(summary.loc[exp, 'delta_r2_vs_core'])}; "
            f"ΔMAE = {fmt(summary.loc[exp, 'delta_mae_vs_core'])}; "
            f"ΔRMSE = {fmt(summary.loc[exp, 'delta_rmse_vs_core'])}."
        )

core_year = year_summary.query("split == 'validation' and experiment == 'ols_core'")
if not core_year.empty:
    drift_range = core_year["mean_residual"].max() - core_year["mean_residual"].min()
    notes.append(
        f"Core residual temporal drift by year: max-min mean residual = {fmt(drift_range)}."
    )

core_quarter = quarter_summary.query("split == 'validation' and experiment == 'ols_core'")
if not core_quarter.empty:
    season_range = core_quarter["mean_residual"].max() - core_quarter["mean_residual"].min()
    notes.append(
        f"Core residual seasonality by quarter: max-min mean residual = {fmt(season_range)}."
    )

DIAGNOSTIC_SUMMARY = "\n".join(f"- {note}" for note in notes)
print(DIAGNOSTIC_SUMMARY)

(TABLE_DIR / "T8_ols_temporal_diagnostic_notes.txt").write_text(DIAGNOSTIC_SUMMARY, encoding="utf-8")



## 11. Outputs written

Tables:

```text
reports/notebook_outputs/ols_temporal_effects/tables/
```

Figures:

```text
reports/notebook_outputs/ols_temporal_effects/figures/
```

Key thesis candidates:

```text
T2_ols_temporal_validation_vs_core.csv
T4_ols_temporal_residual_by_year.csv
T5_ols_temporal_residual_by_quarter.csv
F1_ols_temporal_delta_r2_vs_core.png
F2_ols_temporal_mean_residual_by_year.png
F3_ols_temporal_mean_residual_by_quarter.png
F4_ols_temporal_mean_residual_by_year_quarter.png
```
